                # 04 Model Validation Diagnostics Lab

                Purpose: pull latest code, run walk-forward validation,
                falsification placeholders, drift diagnostics, and the Phase 1
                gate. The output decides whether research improves, freezes a
                candidate later, or remains blocked.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Validation And Gate Configs


In [ ]:
                from pathlib import Path
                for rel in ["configs/validation.yaml", "configs/performance_gate.yaml"]:
                    path = Path(PROJECT_ROOT) / rel
                    print("\n---", rel, "---")
                    print(path.read_text())
                


## 5. Run Walk-Forward Validation


In [ ]:
walk_forward_result = run_stage_checked("validate_walk_forward", config_name="validation")


## 6. Run Falsification Checks


In [ ]:
falsification_result = run_stage_checked("run_falsification", config_name="validation")


## 7. Run Drift Checks


In [ ]:
drift_result = run_stage_checked("run_drift_checks", config_name="validation")


## 8. Evaluate Phase 1 Gate


In [ ]:
gate_result = run_stage_checked("evaluate_phase1_gate", config_name="performance_gate")


## 9. Inspect Validation Outputs


In [ ]:
                import json
                from pathlib import Path
                import pandas as pd

                experiment_dir = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID
                print("\nFold metrics")
                print(pd.read_csv(experiment_dir / "fold_metrics.csv").head())
                print("\nDecision report")
                pprint(json.loads((experiment_dir / "decision_report.json").read_text()))
                


## 10. Confirm Phase 2 Is Still Blocked


In [ ]:
                assert gate_result["phase2_allowed"] is False
                assert gate_result["decision"] == "DO_NOT_PROCEED_TO_PHASE2"
                print("Phase 2 remains blocked unless Future-OOS promotion rules pass later.")
                


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
